# CineIQ — Weighted Ensemble Recommender

Individual recommenders have complementary strengths and weaknesses.
We combine them into a **weighted ensemble** for more robust recommendations.

## Weights
- **SVD (Collaborative)**: 50% — strongest signal from user behavior
- **Content (TF-IDF)**: 30% — ensures genre relevance
- **Popularity**: 20% — fallback for new/niche items

Each component is normalized to [0, 1] before weighting.

In [1]:
# Imports
import pandas as pd
import numpy as np
import pickle

# Load everything
movies = pd.read_csv('../data/processed/movies.csv')
ratings = pd.read_csv('../data/processed/merged.csv')
cosine_sim = pickle.load(open('../models/cosine_sim.pkl', 'rb'))
indices = pickle.load(open('../models/indices.pkl', 'rb'))
svd = pickle.load(open('../models/svd_model.pkl', 'rb'))
print("All loaded.")

All loaded.


## Loading Pre-trained Artifacts
We load the cosine similarity matrix, SVD model, and index mapping from the previous notebooks.

In [2]:
# Content-based scores for a movie
def content_scores(title, n=20):
    if title not in indices:
        return {}
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:n+1]
    return {movies.iloc[i[0]]['movieId']: i[1] for i in sim_scores}

## Content Scores
For a given movie, retrieve the top-50 most similar movies by cosine similarity on genre features.

In [3]:
# SVD scores for a user
def svd_scores(user_id, movie_ids):
    return {mid: svd.predict(user_id, mid).est for mid in movie_ids}

# Popularity scores (global avg rating)
popularity = ratings.groupby('movieId')['rating'].mean().to_dict()

def popularity_scores(movie_ids):
    max_r = max(popularity.values())
    return {mid: popularity.get(mid, 0) / max_r for mid in movie_ids}

## SVD & Popularity Scores
- **SVD**: Predict ratings for candidate movies using the trained matrix factorization model.
- **Popularity**: Normalized global average rating — well-liked movies score higher.

In [4]:
# Ensemble
# Cell 6 — Ensemble (fixed)
def ensemble_recommend(user_id, liked_movie_title, n=10,
                        w_content=0.3, w_svd=0.5, w_pop=0.2):
    candidates = content_scores(liked_movie_title, n=50)
    if not candidates:
        return "Movie not found."
    
    movie_ids = list(candidates.keys())
    
    max_c = max(candidates.values()) or 1
    c_scores = {mid: v/max_c for mid, v in candidates.items()}
    
    s_raw = svd_scores(user_id, movie_ids)
    max_s = max(s_raw.values()) or 1
    s_scores = {mid: v/max_s for mid, v in s_raw.items()}
    
    p_scores = popularity_scores(movie_ids)
    
    final = {}
    for mid in movie_ids:
        final[mid] = (w_content * c_scores.get(mid, 0) +
                      w_svd    * s_scores.get(mid, 0) +
                      w_pop    * p_scores.get(mid, 0))
    
    top_ids = sorted(final, key=final.get, reverse=True)[:n]
    result = movies[movies['movieId'].isin(top_ids)][['movieId', 'title', 'genres']].copy()
    result['score'] = result['movieId'].map(final)
    return result.sort_values('score', ascending=False)[['title', 'genres', 'score']]

## Ensemble Function
`ensemble_recommend(user_id, liked_movie_title, n=10)` builds candidate set from content similarity, scores each by all three methods, and combines via weighted sum.

In [5]:
# Test
print(ensemble_recommend(user_id=1, liked_movie_title="Toy Story (1995)"))

# Save weights config
import json
weights = {"w_content": 0.3, "w_svd": 0.5, "w_pop": 0.2}
json.dump(weights, open('../models/ensemble_weights.json', 'w'))
print("Saved.")

                                            title  \
3045                           Toy Story 2 (1999)   
2286                         Bug's Life, A (1998)   
1205                      Grand Day Out, A (1992)   
1132                   Wrong Trousers, The (1993)   
1010  Winnie the Pooh and the Blustery Day (1968)   
1838                                 Mulan (1998)   
2068                       Charlotte's Web (1973)   
3682                           Chicken Run (2000)   
2070                   Secret of NIMH, The (1982)   
2965                            Robin Hood (1973)   

                           genres     score  
3045  Animation|Children's|Comedy  0.968757  
2286  Animation|Children's|Comedy  0.913858  
1205             Animation|Comedy  0.909927  
1132             Animation|Comedy  0.904065  
1010         Animation|Children's  0.898150  
1838         Animation|Children's  0.882837  
2068         Animation|Children's  0.879243  
3682  Animation|Children's|Comedy  0.877765  
20

## Test Run & Save
Testing with user_id=1 and "Toy Story (1995)" returns a diverse set of animated films ranked by hybrid score.
We save the ensemble weights for the API server to use.